### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="video_game_fps_prediction",
    dataset_year="2020",
    domain_str="technology & internet",
    # Data Source
    dataset_source="OpenML",
    # Also https://github.com/svpeeters/performance_prediction
    original_dataset_source_download_link="https://www.openml.org/d/44992",
    download_description="""
wget https://api.openml.org/data/download/22111856/file22f1639d20997.arff \
&& mkdir -p local-data-warehouse/video_game_fps_prediction \
&& mv file22f1639d20997.arff local-data-warehouse/video_game_fps_prediction/
""",
    # References
    academic_reference_bibtex="""@inproceedings{peeters2021performance,
  title={Performance Prediction for Hardware-Software Configurations: A Case Study for Video Games},
  author={Peeters, Sven and Melnikov, Vitalik and H{\""u}llermeier, Eyke},
  booktitle={International Symposium on Intelligent Data Analysis},
  pages={222--234},
  year={2021},
  organization={Springer}
}
""",
    academic_reference_bibtex_key="peeters2021performance",
    license="CC BY", # That is all there is on OpenML...
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We start with the version from OpenML.

- Following the original work, we do not use CPU and GPU name as features to learn to generalize across the features of CPUs and GPUs.
- The OpenML version is only data from the FPS benchmark. Thus, we perform on a subset of data points from the original study.
- We diverge in the way we split from the original work. The task could be split on games (predict FPS from past values of other games to a new game), but there are not features to allow generalization across games. Instead, we split on hardware configuration (CPU-GPU combination). That is, the task is to be given a set of hardware configurations, for which we have values for a set of games, and then we want to predict the performance of a new hardware configuration on the same set of games. The features in the dataset allow this kind of prediction task and it makes sense to use to save on benchmark costs and interpolate missing hardware combinations -- a surrogate model for hardware performance. Along these lines, it might also be a valid task when one assume we have samples of some hardware configurations for some games and want to interpolate the rest. But predicting for a new, unseen hardware configuration seems more realistic.
- The data has measurement for two cases, the game on medium and the game on max settings. It is unclear if one would build one or two models for this case. In theory, this could also be seen as two labels. We drop the medium setting only keep one task, as this is a less ambiguous task definition.
- We normalize the FPS to ensure that all target values are on the same scale. By default, FPS values can be in vastly different scales depending on the game, having different distributions. Since, we care about comparing hardware, we can simply predict performance relative to a baseline, as in, how much better is the hardware than some default hardware setting. We normalize and then drop this configuration. In other words, one assumes we always have benchmarked the FPS values for this configuration and all others could be potentially a target during one of the splits. We normalizes with respect to "NVIDIA GeForce GTX 1060 5 GB" and "Intel Core i5-8400". We compute the relative ratio to this baseline configuration. We then log scale this ratio.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="log_fps_ratio",
    problem_type="regression",
    objective_metric_name="rmse",
    # For grouped data
    group_on="HardwareConfigId",
    group_labels="per_sample",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import arff
import uuid

with open(dataset_mold.path / "file22f1639d20997.arff", "r") as f:
    arff_data = arff.load(f)
df = pd.DataFrame(arff_data["data"], columns=[col[0] for col in arff_data["attributes"]])
print("Loaded data shape:", df.shape)
df = df[df["GameSetting"] == "max"] # Keep only max settings, drop medium settings

df["HardwareConfigId"] = df["CpuName"] + "_" + df["GpuName"]
hardware_config_baseline = "Intel Core i5-8400_NVIDIA GeForce GTX 1060 5 GB"

# Normalize target
baseline_map = (
    df[df["HardwareConfigId"] == hardware_config_baseline]
    .set_index("GameName")["FPS"]
)
# Broadcast baseline FPS to all rows
df["baseline_fps"] = df["GameName"].map(baseline_map)
# Normalize
df["fps_ratio"] = df["FPS"] / df["baseline_fps"]
df["log_fps_ratio"] = np.log(df["fps_ratio"])
df = df[df["HardwareConfigId"] != hardware_config_baseline]
# Map to ID to avoid leaking from names (which we do not want to use)
mapping = {val: uuid.uuid4().hex[:12] for val in df["HardwareConfigId"].unique()}
df["HardwareConfigId"] = df["HardwareConfigId"].map(mapping)


df = df.drop(columns=[
    "CpuName",
    "GpuName",
    "GameSetting",
    # All empty or constant
    "GpuNumberOfExecutionUnits",
    "CpuBaseClock",
    # Other target features
    "FPS", "baseline_fps", "fps_ratio",
])

as_cat_type = ["GpuVulkan", "GameName", "GpuShaderModel", "GpuOpenCL", "GpuOpenGL", "GpuMemoryType", "GpuDirectX","GpuBus.interface", "GpuArchitecture", "CpuMultiplierUnlocked", "HardwareConfigId"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle data

Loaded data shape: (24624, 44)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 12,288
Columns: 40
Use sampling: False (sample size: 12,288)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['HardwareConfigId', 'GpuTextureRate', 'GpuPixelRate', 'GpuFP32Performance', 'GameName', 'GpuBoostClock', 'GpuBaseClock', 'GpuNumberOfTMUs', 'GpuBandwidth', 'GpuNumberOfShadingUnits']
Rows remaining as candidates after top-10 filter: 0 (of 12,288)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,CpuNumberOfCores,CpuNumberOfThreads,CpuCacheL1,CpuCacheL2,CpuCacheL3,CpuDieSize,CpuFrequency,CpuMultiplier,CpuMultiplierUnlocked,CpuProcessSize,CpuTDP,CpuNumberOfTransistors,CpuTurboClock,GpuArchitecture,GpuBandwidth,GpuBaseClock,GpuBoostClock,GpuBus.interface,GpuNumberOfComputeUnits,GpuDieSize,GpuDirectX,GpuFP32Performance,GpuMemoryBus,GpuMemorySize,GpuMemoryType,GpuOpenCL,GpuOpenGL,GpuPixelRate,GpuProcessSize,GpuNumberOfROPs,GpuShaderModel,GpuNumberOfShadingUnits,GpuNumberOfTMUs,GpuTextureRate,GpuNumberOfTransistors,GpuVulkan,GameName,GameResolution,HardwareConfigId,log_fps_ratio
0,4.0,4.0,256.0,1024.0,6.0,NaN,3400.0,34.0,0,14.0,65.0,NaN,3800.0,Turing,448000.0,1515.0,1710.0,PCIe 3.0 x16,NaN,0.000545,12 Ultimate,10070000.0,256.0,8000.0,GDDR6,1.2,4.6,109400.0,12.0,64.0,6.5,2944.0,184.0,314600.0,13600.0,1.2.131,battlefield4,1080.0,8fe96644ac0a,0.420369
1,4.0,8.0,256.0,1024.0,8.0,NaN,4200.0,42.0,1,14.0,91.0,NaN,4500.0,Pascal,192200.0,1506.0,1709.0,PCIe 3.0 x16,NaN,0.000314,12,4375000.0,192.0,6000.0,GDDR5X,1.2,4.6,82030.0,16.0,48.0,6.4,1280.0,80.0,136700.0,7200.0,1.2.131,worldOfTanks,1080.0,3d6f12bc6f28,0.172534
2,6.0,12.0,384.0,1536.0,12.0,NaN,3700.0,37.0,1,14.0,95.0,NaN,4700.0,Turing,288000.0,1500.0,1770.0,PCIe 3.0 x16,NaN,0.000284,12,5437000.0,192.0,6000.0,GDDR6,1.2,4.6,84960.0,12.0,48.0,6.5,1536.0,96.0,169900.0,6600.0,1.2.131,seaOfThieves,1080.0,3550f1efae38,0.262627
3,6.0,12.0,576.0,3072.0,32.0,0.000074,3600.0,36.0,1,7.0,65.0,3800.0,4200.0,Turing,336000.0,1365.0,1680.0,PCIe 3.0 x16,NaN,0.000445,12 Ultimate,6451000.0,192.0,6000.0,GDDR6,1.2,4.6,80640.0,12.0,48.0,6.5,1920.0,120.0,201600.0,10800.0,1.2.131,farCry5,1080.0,4f8a33ce261b,0.351190
4,6.0,6.0,384.0,1536.0,9.0,NaN,3600.0,36.0,1,14.0,95.0,NaN,4300.0,Pascal,256300.0,1506.0,1683.0,PCIe 3.0 x16,NaN,0.000314,12,6463000.0,256.0,8000.0,GDDR5X,1.2,4.6,107700.0,16.0,64.0,6.4,1920.0,120.0,202000.0,7200.0,1.2.131,battlefield4,1080.0,e8eefebf753b,0.217723


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,CpuMultiplierUnlocked,category,0.0,0.00,2.0,"1, 0"
1,GpuArchitecture,category,0.0,0.00,6.0,"Turing, Pascal, Maxwell 2.0, GCN 4.0, RDNA 1.0, GCN 5.0"
2,GpuBus.interface,category,0.0,0.00,2.0,"PCIe 3.0 x16, PCIe 4.0 x16"
3,GpuDirectX,category,0.0,0.00,2.0,"12, 12 Ultimate"
4,GpuMemoryType,category,0.0,0.00,4.0,"GDDR5, GDDR6, GDDR5X, HBM2"
5,GpuOpenCL,category,0.0,0.00,2.0,"1.2, 2"
6,GpuOpenGL,category,0.0,0.00,1.0,4.6
7,GpuShaderModel,category,0.0,0.00,2.0,"6.4, 6.5"
8,GpuVulkan,category,0.0,0.00,3.0,"1.2.131, 1.1.126, 1.1.125"
9,GameName,category,0.0,0.00,24.0,"aWayOut, airMechStrike, apexLegends, battlefield4, battletech, callOfDutyWW2, counterStrikeGlobalOffensive, destiny2, dota2, farCry5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
CpuNumberOfCores,12288.0,6.316406e+00,2.080848e+00,2.000000e+00,1.200000e+01
CpuNumberOfThreads,12288.0,1.085156e+01,5.204762e+00,4.000000e+00,2.400000e+01
CpuCacheL1,12288.0,5.156250e+02,2.336821e+02,1.280000e+02,1.152000e+03
CpuCacheL2,12288.0,2.508000e+03,1.399434e+03,5.120000e+02,6.144000e+03
CpuCacheL3,12288.0,1.822852e+01,1.423826e+01,3.000000e+00,6.400000e+01
CpuDieSize,5832.0,1.264444e-04,5.863970e-05,7.400000e-05,1.920000e-04
CpuFrequency,12288.0,3.591016e+03,3.161130e+02,2.800000e+03,4.200000e+03
CpuMultiplier,12288.0,3.591016e+01,3.161130e+00,2.800000e+01,4.200000e+01
CpuProcessSize,12288.0,1.194336e+01,3.018736e+00,7.000000e+00,1.400000e+01
CpuTDP,12288.0,8.387891e+01,1.654326e+01,5.100000e+01,1.050000e+02


In [7]:
# Categorical Feature Statistics
cat_stats

value  count     pct
column                rank                              
CpuMultiplierUnlocked 1                 1   9720   79.10
                      2                 0   2568   20.90
GameName              1           aWayOut    512    4.17
                      2     airMechStrike    512    4.17
                      3       apexLegends    512    4.17
                      4      battlefield4    512    4.17
                      5        battletech    512    4.17
GpuArchitecture       1            Turing   4104   33.40
                      2            Pascal   3624   29.49
                      3       Maxwell 2.0   1824   14.84
                      4           GCN 4.0   1368   11.13
                      5          RDNA 1.0    912    7.42
GpuBus.interface      1      PCIe 3.0 x16  11376   92.58
                      2      PCIe 4.0 x16    912    7.42
GpuDirectX            1                12   9552   77.73
                      2       12 Ultimate   2736   22.27
GpuMemoryType         1             GDDR5   5448   44.34
                      2             GDDR6   4560   37.11
                      3            GDDR5X   1824   14.84
                      4              HBM2    456    3.71
GpuOpenCL             1               1.2   9096   74.02
                      2                 2   3192   25.98
GpuOpenGL             1               4.6  12288  100.00
GpuShaderModel        1               6.4   7272   59.18
                      2               6.5   5016   40.82
GpuVulkan             1           1.2.131  10008   81.45
                      2           1.1.126   1824   14.84
                      3           1.1.125    456    3.71
HardwareConfigId      1      eb59338f4e8d     24    0.20
                      2      eb9063086748     24    0.20
                      3      ebe1aa033895     24    0.20
                      4      ec3c148dbc73     24    0.20
                      5      ec5fccac13c6     24    0.20

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,20.69,-0.634,-1.374,0.078,0.073,log1p,-2630.0,683827.5,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using label-per-sample grouped splits.


Repeat 0, Fold 0:
            Train N: 8184, Test N: 4104
            Target Distribution:
            	Train target distribution: 0.21040359102913025
            	Test target distribution: 0.1990666293205664
            Group Distribution HardwareConfigId:
            	Train: 341
            	Test: 171
            
Repeat 0, Fold 1:
            Train N: 8184, Test N: 4104
            Target Distribution:
            	Train target distribution: 0.2153224069897766
            	Test target distribution: 0.1892577624048916
            Group Distribution HardwareConfigId:
            	Train: 341
            	Test: 171
            
Repeat 0, Fold 2:
            Train N: 8208, Test N: 4080
            Target Distribution:
            	Train target distribution: 0.194162195862729
            	Test target distribution: 0.23167380688057032
            Group Distribution HardwareConfigId:
            	Train: 342
            	Test: 170
            
Repeat 1, Fold 0:
            Train N: 8184, Tes

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to video_game_fps_prediction/019d7392-6ce5-72bc-b598-193597825a66


019d7392-6ce5-72bc-b598-193597825a66
efc5038d6ea680e83d270a2f7dfce7e8711d6cf4e148c0a342b6ce97c6f5fcc7
